Note: you may need to restart the kernel to use updated packages.


✅ GPU (MPS) is available! Using GPU for training.
   This will significantly speed up training on Mac M4.

Device selected: mps
MPS built: True
PyTorch version: 2.9.0


In [4]:
# Import Libraries
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from ultralytics import YOLO
import random
import time
import pandas as pd
import yaml


In [5]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="B0NQSUPe9ENM4O6rTvIW")
project = rf.workspace("nycu-wnnjh").project("acne04_new")
version = project.version(2)
dataset = version.download("yolov8")
                

loading Roboflow workspace...
loading Roboflow project...


## 1. Data Exploration


In [6]:
# Sử dụng dataset từ Roboflow
# Dataset đã được download ở cell 5

# Lấy đường dẫn dataset
data_yaml_path = os.path.join(dataset.location, 'data.yaml')
print(f"✅ Dataset location: {dataset.location}")
print(f"✅ Data YAML path: {data_yaml_path}")

# Kiểm tra cấu trúc dataset
train_path = os.path.join(dataset.location, 'train')
val_path = os.path.join(dataset.location, 'valid')
test_path = os.path.join(dataset.location, 'test')

# Count images and labels
if os.path.exists(train_path):
    train_images = os.listdir(os.path.join(train_path, 'images')) if os.path.exists(os.path.join(train_path, 'images')) else []
    train_labels = os.listdir(os.path.join(train_path, 'labels')) if os.path.exists(os.path.join(train_path, 'labels')) else []
else:
    train_images = []
    train_labels = []

if os.path.exists(val_path):
    val_images = os.listdir(os.path.join(val_path, 'images')) if os.path.exists(os.path.join(val_path, 'images')) else []
    val_labels = os.listdir(os.path.join(val_path, 'labels')) if os.path.exists(os.path.join(val_path, 'labels')) else []
else:
    val_images = []
    val_labels = []

if os.path.exists(test_path):
    test_images = os.listdir(os.path.join(test_path, 'images')) if os.path.exists(os.path.join(test_path, 'images')) else []
    test_labels = os.listdir(os.path.join(test_path, 'labels')) if os.path.exists(os.path.join(test_path, 'labels')) else []
else:
    test_images = []
    test_labels = []

print(f'\n📊 Dataset Statistics:')
print(f'Training images: {len(train_images)}, labels: {len(train_labels)}')
print(f'Validation images: {len(val_images)}, labels: {len(val_labels)}')
print(f'Test images: {len(test_images)}, labels: {len(test_labels)}')

# Đọc data.yaml để xem số classes
if os.path.exists(data_yaml_path):
    import yaml
    with open(data_yaml_path, 'r') as f:
        data_config = yaml.safe_load(f)
    print(f'\n📋 Dataset Config:')
    print(f'   Classes: {data_config.get("nc", "N/A")}')
    print(f'   Class names: {data_config.get("names", "N/A")}')


✅ Dataset location: /Users/quangthai/Documents/AI in Bioinfomatics/acne2/Acne04_new-2
✅ Data YAML path: /Users/quangthai/Documents/AI in Bioinfomatics/acne2/Acne04_new-2/data.yaml

📊 Dataset Statistics:
Training images: 2982, labels: 2982
Validation images: 283, labels: 283
Test images: 142, labels: 142

📋 Dataset Config:
   Classes: 1
   Class names: ['fore']


## 2. Hyperparameter Configuration - TỐI ƯU ACCURACY


In [7]:
# Track training start time
training_start_time = time.time()

# Sử dụng dataset từ Roboflow
data_yaml_path = os.path.join(dataset.location, 'data.yaml')
print(f"📁 Using dataset: {data_yaml_path}")

# Train với config TỐI ƯU ACCURACY
results = model.train(
    data=data_yaml_path,
    
    # === Epochs & Early Stopping ===
    epochs=TRAIN_CONFIG['epochs'],
    patience=TRAIN_CONFIG['patience'],
    
    # === Image Size ===
    imgsz=TRAIN_CONFIG['imgsz'],
    
    # === Batch Size ===
    batch=TRAIN_CONFIG['batch'],
    
    # === Learning Rate ===
    lr0=TRAIN_CONFIG['lr0'],
    lrf=TRAIN_CONFIG['lrf'],
    momentum=0.937,
    weight_decay=0.0005,
    
    # === Warmup ===
    warmup_epochs=TRAIN_CONFIG['warmup_epochs'],
    warmup_momentum=TRAIN_CONFIG['warmup_momentum'],
    warmup_bias_lr=TRAIN_CONFIG['warmup_bias_lr'],
    
    # === Augmentation ===
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=TRAIN_CONFIG['degrees'],
    translate=0.1,
    scale=0.5,
    shear=TRAIN_CONFIG['shear'],
    perspective=0.0,
    flipud=0.0,
    fliplr=0.5,
    mosaic=TRAIN_CONFIG['mosaic'],
    mixup=TRAIN_CONFIG['mixup'],
    copy_paste=TRAIN_CONFIG.get('copy_paste', 0.0),
    auto_augment='randaugment',
    erasing=0.4,
    
    # === Loss Weights ===
    box=TRAIN_CONFIG.get('box', 7.5),
    cls=TRAIN_CONFIG.get('cls', 0.5),
    dfl=TRAIN_CONFIG.get('dfl', 1.5),
    
    # === Multi-scale ===
    multi_scale=TRAIN_CONFIG.get('multi_scale', False),
    
    # === Close Mosaic ===
    close_mosaic=TRAIN_CONFIG.get('close_mosaic', 10),
    
    # === Tối ưu ===
    device=device,
    workers=8,
    amp=True,
    save=True,
    plots=True,
    cache=TRAIN_CONFIG.get('cache', False),
    
    # === Learning Rate Schedule ===
    cos_lr=TRAIN_CONFIG['cos_lr'],
    
    # === Optimizer ===
    optimizer='AdamW',
    
    # === Validation ===
    val=True,
    split='val',
    
    # === Name ===
    name='acne_detection_roboflow',
)

training_time = time.time() - training_start_time

print("\n✅ Training completed!")
print(f"⏱️  Thời gian training: {training_time/60:.1f} phút ({training_time/3600:.2f} giờ)")
print(f"📊 Best model: {results.save_dir}/weights/best.pt")
print(f"📊 Last model: {results.save_dir}/weights/last.pt")


📁 Using dataset: /Users/quangthai/Documents/AI in Bioinfomatics/acne2/Acne04_new-2/data.yaml


NameError: name 'model' is not defined

✅ Model initialized: yolov8s.pt
   Model ready for fast training with optimized parameters!


✅ MPS cache cleared


Ultralytics 8.3.237 🚀 Python-3.11.13 torch-2.9.0 MPS (Apple M4 Pro)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=data-2/data.yaml, degrees=10.0, deterministic=True, device=mps, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=150, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=512, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.02, lrf=0.1, mask_ratio=4, max_det=300, mixup=0.1, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=acne_detection_fast2, nbs=64, nms=False, opset=None, optimize=False, optimizer=AdamW, overlap_mask=True, patience=30, perspective=0.0, plots=True, pose=12.0, pretraine

In [ ]:
# Evaluate on test set (using GPU if available)
data_yaml_path = os.path.join(dataset.location, 'data.yaml')
metrics = model.val(data=data_yaml_path, split='test', device=device)
print(f"\n📊 Test Results:")
print(f"   mAP50: {metrics.box.map50:.4f} ({metrics.box.map50*100:.2f}%)")
print(f"   mAP50-95: {metrics.box.map:.4f} ({metrics.box.map*100:.2f}%)")
print(f"   Precision: {metrics.box.mp:.4f} ({metrics.box.mp*100:.2f}%)")
print(f"   Recall: {metrics.box.mr:.4f} ({metrics.box.mr*100:.2f}%)")

# Check if target achieved
if metrics.box.map50 > 0.2:
    print("\n🎉 SUCCESS! Đạt trên 20% mAP50!")
    if metrics.box.map50 > 0.5:
        print("✅ Tuyệt vời! Đạt trên 50% mAP50")
    if metrics.box.map50 > 0.7:
        print("🎉 Xuất sắc! Đạt trên 70% mAP50")
else:
    print("\n⚠️ Cần cải thiện thêm")
